In [ ]:
!pip install chess pandas tensorflow zstandard 

In [ ]:
!wget https://database.lichess.org/lichess_db_puzzle.csv.zst

In [ ]:
import zstandard as zstd

def decompress_zst(input_file, output_file):
    dctx = zstd.ZstdDecompressor()
    with open(input_file, 'rb') as ifh, open(output_file, 'wb') as ofh:
        dctx.copy_stream(ifh, ofh)

decompress_zst('lichess_db_puzzle.csv.zst', 'lichess_db_puzzle.csv')

In [ ]:
import os

file_path = 'lichess_db_puzzle.csv'
if os.path.exists(file_path):
    print(f"File exists and its size is {os.path.getsize(file_path) / (1024*1024):.2f} MB")
else:
    print("File does not exist or is not accessible")


In [ ]:
import pandas as pd

def peek_csv(file_path, num_lines=5):
    df = pd.read_csv(file_path, nrows=num_lines)
    print(df.head())

peek_csv('lichess_db_puzzle.csv')


In [ ]:
import chess
import numpy as np
import pandas as pd

def fen_to_matrix(fen):
    board = chess.Board(fen)
    matrix = np.zeros((8, 8, 21), dtype=np.float32)

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        row, col = divmod(square, 8)
        if piece:
            piece_type = piece.piece_type - 1
            if piece.color == chess.BLACK:
                piece_type += 6
            matrix[row, col, piece_type] = 1
        else:
            matrix[row, col, 12] = 1

    matrix[:, :, 13] = float(board.turn == chess.WHITE)
    matrix[:, :, 14] = (
        (board.has_kingside_castling_rights(chess.WHITE) * 1) +
        (board.has_queenside_castling_rights(chess.WHITE) * 2) +
        (board.has_kingside_castling_rights(chess.BLACK) * 4) +
        (board.has_queenside_castling_rights(chess.BLACK) * 8)
    ) / 15

    if board.ep_square:
        ep_row, ep_col = divmod(board.ep_square, 8)
        matrix[ep_row, ep_col, 15] = 1

    matrix[:, :, 16] = board.halfmove_clock / 100
    matrix[:, :, 17] = (board.fullmove_number - 1) / 500
    matrix[:, :, 18] = float(board.is_check())

    for move in board.legal_moves:
        to_row, to_col = divmod(move.to_square, 8)
        matrix[to_row, to_col, 19] += 1
    matrix[:, :, 19] /= np.max(matrix[:, :, 19]) + 1e-8

    for square in chess.SQUARES:
        row, col = divmod(square, 8)
        white_attackers = board.attackers(chess.WHITE, square)
        black_attackers = board.attackers(chess.BLACK, square)
        matrix[row, col, 20] = (len(white_attackers) - len(black_attackers)) / 8

    return matrix

def preprocess_puzzle_csv(csv_file, max_puzzles=100000):
    positions = []
    evaluations = []
    df = pd.read_csv(csv_file)

    # Drop rows with NaN values
    df.dropna(subset=['FEN', 'Moves'], inplace=True)

    print(f"Starting to process {csv_file}")

    for i, row in df.iterrows():
        if i >= max_puzzles:
            break
        try:
            fen = row['FEN']
            moves = row['Moves'].split()
            if not moves:
                continue
            first_move = moves[0]

            board = chess.Board(fen)
            matrix = fen_to_matrix(fen)
            positions.append(matrix)

            if board.turn == chess.WHITE:
                evaluations.append(1.0)  # White to move, good move is +1
            else:
                evaluations.append(-1.0)  # Black to move, good move is -1

            if len(positions) % 1000 == 0:
                print(f"Added {len(positions)} valid positions")

        except Exception as e:
            print(f"Error processing puzzle {i}: {str(e)}")
            continue

    print(f"Finished processing. Found {len(positions)} valid positions.")

    positions_array = np.array(positions)
    evaluations_array = np.array(evaluations)

    np.save('positions.npy', positions_array)
    np.save('evaluations.npy', evaluations_array)

    print(f"Saved positions array with shape: {positions_array.shape}")
    print(f"Saved evaluations array with shape: {evaluations_array.shape}")

    return positions_array, evaluations_array

positions, evaluations = preprocess_puzzle_csv('lichess_db_puzzle.csv', max_puzzles=100000)

In [ ]:
from sklearn.model_selection import train_test_split

positions = np.load('positions.npy')
evaluations = np.load('evaluations.npy')

print(f"Loaded positions array with shape: {positions.shape}")
print(f"Loaded evaluations array with shape: {evaluations.shape}")

positions_reshaped = positions.reshape(-1, 8, 8, 21)

X_train, X_val, y_train, y_val = train_test_split(positions_reshaped, evaluations, test_size=0.2, random_state=42)


In [ ]:
def augment_data(X, y):
    augmented_X = []
    augmented_y = []

    for board, evaluation in zip(X, y):
        augmented_X.append(board)
        augmented_y.append(evaluation)

        flipped_board = np.flip(board, axis=1)
        flipped_board[:,:,0:6], flipped_board[:,:,6:12] = flipped_board[:,:,6:12], flipped_board[:,:,0:6].copy()
        flipped_board[:,:,13] = 1 - flipped_board[:,:,13]
        flipped_board[:,:,14], flipped_board[:,:,15] = flipped_board[:,:,15], flipped_board[:,:,14].copy()
        augmented_X.append(flipped_board)
        augmented_y.append(-evaluation)

        rotated_board = np.rot90(board, k=2, axes=(0, 1))
        augmented_X.append(rotated_board)
        augmented_y.append(evaluation)

        flipped_rotated_board = np.rot90(flipped_board, k=2, axes=(0, 1))
        augmented_X.append(flipped_rotated_board)
        augmented_y.append(-evaluation)

    return np.array(augmented_X), np.array(augmented_y)


In [ ]:
import matplotlib.pyplot as plt

def plot_chess_board(board, evaluation, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xticks(np.arange(8))
    ax.set_yticks(np.arange(8))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(which='major', color='black', linestyle='-', linewidth=2)

    pieces = ['P', 'N', 'B', 'R', 'Q', 'K', 'p', 'n', 'b', 'r', 'q', 'k']
    colors = ['white', 'black']

    for i in range(8):
        for j in range(8):
            for p, piece in enumerate(pieces):
                if board[i, j, p] == 1:
                    ax.text(j, 7-i, piece, fontsize=20, ha='center', va='center', color=colors[p//6])

    ax.set_title(f"{title}\nEvaluation: {evaluation:.2f}")
    plt.tight_layout()
    plt.show()

sample_index = np.random.randint(0, len(positions_reshaped))
sample_board = positions_reshaped[sample_index]
sample_eval = evaluations[sample_index]

augmented_boards, augmented_evals = augment_data(np.array([sample_board]), np.array([sample_eval]))

plot_chess_board(sample_board, sample_eval, "Original Board")

for i, (board, eval) in enumerate(zip(augmented_boards[1:], augmented_evals[1:]), 1):
    if i == 1:
        title = "Horizontally Flipped Board"
    elif i == 2:
        title = "Rotated Board (180 degrees)"
    else:
        title = "Flipped and Rotated Board"
    plot_chess_board(board, eval, title)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split

def create_cnn_model(input_shape):
    inputs = Input(shape=input_shape)

    x = Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    output = Dense(1, activation='tanh')(x)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mean_squared_error', metrics=['mae'])

    return model

positions = np.load('positions.npy')
evaluations = np.load('evaluations.npy')

print(f"Loaded positions array with shape: {positions.shape}")
print(f"Loaded evaluations array with shape: {evaluations.shape}")

if len(positions.shape) == 2:
    positions_reshaped = positions.reshape(-1, 8, 8, 21)
else:
    positions_reshaped = positions

print(f"Reshaped positions array to shape: {positions_reshaped.shape}")

X_augmented, y_augmented = augment_data(positions_reshaped, evaluations)

print(f"Augmented data shapes: X: {X_augmented.shape}, y: {y_augmented.shape}")

X_train, X_val, y_train, y_val = train_test_split(X_augmented, y_augmented, test_size=0.2, random_state=42)

model = create_cnn_model(input_shape=(8, 8, 21))
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
checkpoint = ModelCheckpoint('model_checkpoint.keras', save_best_only=True, monitor='val_loss')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping, lr_reducer, checkpoint],
    verbose=1
)

train_score = model.evaluate(X_train, y_train, verbose=0)
val_score = model.evaluate(X_val, y_val, verbose=0)

print(f"Train Loss: {train_score[0]:.4f}, Train MAE: {train_score[1]:.4f}")
print(f"Val Loss: {val_score[0]:.4f}, Val MAE: {val_score[1]:.4f}")

model.save('chess_evaluation_model_cnn.keras')


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['lr'], label='Learning Rate')
plt.title('Learning Rate')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()

plt.tight_layout()
plt.show()
